# 02 — Modeling & SHAP Explainability
Credit Risk Predictor: XGBoost training, evaluation, and SHAP interpretability

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap

plt.rcParams['figure.dpi'] = 120
%matplotlib inline

## Load & Preprocess

In [ ]:
df = pd.read_csv('../data/credit_risk_data.csv')
print(f'Shape: {df.shape}, Default rate: {df.loan_status.eq("Default").mean():.2%}')

In [ ]:
X = df.drop(columns=['loan_status', 'num_delinquencies'])
y = (df['loan_status'] == 'Default').astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print(f'Train: {X_train_res.shape}, Test: {X_test_scaled.shape}')
print(f'After SMOTE default rate: {y_train_res.mean():.2%}')

## Train XGBoost

In [ ]:
model = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    reg_lambda=1.0, reg_alpha=0.1,
    eval_metric='logloss', random_state=42
)
model.fit(X_train_res, y_train_res)
print('Training complete')

## Evaluate

In [ ]:
y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]
print(f'ROC-AUC: {roc_auc_score(y_test, y_proba):.3f}')
print()
print(classification_report(y_test, y_pred, target_names=['Fully Paid', 'Default']))

In [ ]:
import seaborn as sns
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Fully Paid', 'Default'], yticklabels=['Fully Paid', 'Default'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted');

## Save Model

In [ ]:
joblib.dump(model, '../models/xgb_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
print('Model and scaler saved to ../models/')

## SHAP Explainability
### Global Feature Importance

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_test_scaled)

plt.figure()
shap.plots.bar(shap_values, max_display=12, show=False)
plt.title('Global SHAP Feature Importance')
plt.tight_layout();

### Beeswarm Summary

In [ ]:
plt.figure()
shap.plots.beeswarm(shap_values, max_display=12, show=False)
plt.title('SHAP Beeswarm — Feature Impact Distribution')
plt.tight_layout();

### Waterfall — Default Risk Applicant

In [ ]:
high_risk_idx = y_test[y_test == 1].index[0]
high_risk_row = X_test.loc[[high_risk_idx]]
high_risk_explanation = explainer(scaler.transform(high_risk_row))
high_risk_explanation.feature_names = list(X.columns)

shap.plots.waterfall(high_risk_explanation[0], max_display=12, show=False)
plt.title('SHAP Waterfall — High Risk Applicant')
plt.tight_layout();

### Waterfall — Low Risk Applicant

In [ ]:
low_risk_idx = y_test[y_test == 0].index[0]
low_risk_row = X_test.loc[[low_risk_idx]]
low_risk_explanation = explainer(scaler.transform(low_risk_row))
low_risk_explanation.feature_names = list(X.columns)

shap.plots.waterfall(low_risk_explanation[0], max_display=12, show=False)
plt.title('SHAP Waterfall — Low Risk Applicant')
plt.tight_layout();

### Save Explainer for Dashboard

In [ ]:
joblib.dump(explainer, '../models/shap_explainer.pkl')
print('SHAP explainer saved')

## Summary
- XGBoost with SMOTE handles 22.78% default rate
- ROC-AUC ~0.92, Default F1 ~0.72
- Top drivers: credit_score, dti_ratio, delinquent_history
- Higher credit score → negative SHAP → pushes toward approval